# Gravitational-Wave BayesFlow — Full Colab Pipeline

Run this notebook from top to bottom. It combines dataset generation, model training, and diagnostics for the final 25,000-simulation run. Files are saved directly to Google Drive. The first run restarts the Colab session once after installing packages; after it reconnects, choose **Run all** again.

Before starting, select **Runtime → Change runtime type → GPU**. The simulator mainly uses the CPU, while training uses the GPU.

## 1. Install packages and connect Drive

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab may already have NumPy loaded before pip replaces it. Restarting once
# prevents a mixture of old in-memory modules and new files on disk.
SETUP_MARKER = Path("/content/.gw_packages_ready")

if not SETUP_MARKER.exists():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--no-cache-dir",
        "numpy==2.3.5", "scipy==1.16.3",
        "pycbc", "bayesflow==2.0.12", "matplotlib", "tqdm",
    ])
    SETUP_MARKER.touch()
    print("Packages installed. Colab will restart once; then click Run all again.")
    os.kill(os.getpid(), 9)

print("Package setup is ready.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import sys
import time
import warnings

os.environ["KERAS_BACKEND"] = "torch"
warnings.filterwarnings("ignore", message="Wswiglal-redir-stdio.*")

PROJECT_ROOT = Path("/content/drive/MyDrive/GW_Project")
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Project not found at {PROJECT_ROOT}. Change PROJECT_ROOT to your Drive folder."
    )

project_path = str(PROJECT_ROOT)
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
%cd {PROJECT_ROOT}
print("Project:", PROJECT_ROOT)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from src.dataset import DatasetGenerator
from src.model import (
    PARAMETER_NAMES,
    build_workflow,
    history_to_losses,
    load_npz_dataset,
)
from src.noise import NoiseGenerator
from src.priors import PriorSampler
from src.waveform import (
    F_CROSS,
    F_PLUS,
    N_SAMPLES,
    SAMPLING_RATE,
    generate_waveform,
)
from src.whitening import Whitening

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Signal length:", N_SAMPLES)
print(f"Detector response: F+={F_PLUS:.4f}, Fx={F_CROSS:.4f}")

In [ ]:
N_SIMULATIONS = 25000
EPOCHS = 20
BATCH_SIZE = 64
VALIDATION_FRACTION = 0.10
SEED = 2026

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / f"bayesflow_model_{N_SIMULATIONS}"
FIGURE_DIR = PROJECT_ROOT / "figures"
REPORT_DIR = PROJECT_ROOT / "reports"
DATASET_PATH = DATA_DIR / f"gw_dataset_{N_SIMULATIONS}.npz"
MODEL_PATH = MODEL_DIR / "model.keras"
HISTORY_PATH = REPORT_DIR / f"training_history_{N_SIMULATIONS}.npz"

for directory in (DATA_DIR, MODEL_DIR, FIGURE_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Model:", MODEL_PATH)

## 2. Check one simulated signal

This quick check confirms that the shortened waveform, colored noise, and whitening work before the long dataset generation starts.

In [ ]:
sampler = PriorSampler()
noise_generator = NoiseGenerator()
whitener = Whitening()

example_theta = sampler.sample()
clean_signal = generate_waveform(example_theta)
noisy_signal = noise_generator.add_noise(clean_signal)
whitened_signal = whitener.whiten(noisy_signal)
time_axis = np.arange(N_SAMPLES) / SAMPLING_RATE

assert clean_signal.shape == (2048,)
assert noisy_signal.shape == (2048,)
assert whitened_signal.shape == (2048,)
assert np.isfinite(whitened_signal).all()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, values, title, ylabel in zip(
    axes,
    (clean_signal, noisy_signal, whitened_signal),
    ("Clean waveform", "Signal with colored noise", "Whitened signal"),
    ("Strain", "Strain", "Whitened strain"),
):
    ax.plot(time_axis, values)
    ax.set(title=title, ylabel=ylabel)
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("Time [s]")
fig.tight_layout()
plt.show()
print(example_theta)

## 3. Generate or load the 25,000-simulation dataset

Generation is skipped when the completed dataset already exists in Drive. Do not disconnect the runtime while a new dataset is being generated.

In [ ]:
if DATASET_PATH.exists():
    print("Existing dataset found; generation skipped.")
else:
    start = time.time()
    generator = DatasetGenerator(n_samples=N_SIMULATIONS, output_dir=DATA_DIR)
    generated_X, generated_theta = generator.generate()
    print(f"Generation time: {(time.time() - start) / 60:.1f} minutes")
    del generated_X, generated_theta

with np.load(DATASET_PATH) as saved_data:
    X_shape = saved_data["X"].shape
    theta_shape = saved_data["theta"].shape

assert X_shape == (N_SIMULATIONS, N_SAMPLES)
assert theta_shape == (N_SIMULATIONS, 6)
print("Dataset ready:", X_shape, theta_shape)

## 4. Train the BayesFlow model

The model is trained from scratch with 20,000 training examples. Validation and test each contain 2,500 separate examples.

In [ ]:
from src.model import build_workflow, history_to_losses, load_npz_dataset

# Load the complete 25k dataset
dataset = load_npz_dataset(DATASET_PATH)

# 80% train, 10% validation, 10% test
rng = np.random.default_rng(SEED)
n_samples = len(dataset["parameters"])
indices = rng.permutation(n_samples)

n_test = 2500
n_validation = 2500

test_indices = indices[:n_test]
validation_indices = indices[n_test:n_test + n_validation]
train_indices = indices[n_test + n_validation:]

train_data = {key: value[train_indices] for key, value in dataset.items()}
fit_validation_data = {key: value[validation_indices] for key, value in dataset.items()}
test_data = {key: value[test_indices] for key, value in dataset.items()}

print("Train:", train_data["strain"].shape)
print("Validation:", fit_validation_data["strain"].shape)
print("Test:", test_data["strain"].shape)

assert train_data["strain"].shape[0] == 20000
assert fit_validation_data["strain"].shape[0] == 2500
assert test_data["strain"].shape[0] == 2500

if not torch.cuda.is_available():
    raise RuntimeError("Please enable a GPU runtime before training.")

start = time.time()
workflow = build_workflow(model_dir=MODEL_DIR)
history = workflow.fit_offline(
    data=train_data,
    validation_data=fit_validation_data,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

train_loss, val_loss = history_to_losses(history)
if val_loss is None:
    val_loss = np.array([])

np.savez(HISTORY_PATH, train_loss=train_loss, val_loss=val_loss)

print(f"Training time: {(time.time() - start) / 60:.1f} minutes")
print("Model:", MODEL_PATH, MODEL_PATH.exists())

# Keep the variable used in the original Colab run.
validation_data = test_data
print("Training finished")
print("The following figures will use the test set")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
epochs = np.arange(1, len(train_loss) + 1)
ax.plot(epochs, train_loss, marker="o", label="training")
if val_loss.size:
    ax.plot(epochs, val_loss, marker="o", label="validation")
ax.set(xlabel="Epoch", ylabel="Loss", title="BayesFlow training loss")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "training_loss.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Posterior samples and diagnostics

All diagnostics below use 200 held-out test signals.

In [ ]:
RNG = np.random.default_rng(SEED)
N_DIAGNOSTICS = 200
N_POSTERIOR_SAMPLES = 500
INFERENCE_BATCH_SIZE = 25

indices = RNG.choice(len(test_data["parameters"]), N_DIAGNOSTICS, replace=False)
diagnostic_strain = test_data["strain"][indices]
true_parameters = test_data["parameters"][indices]

posterior_batches = []
for start in range(0, N_DIAGNOSTICS, INFERENCE_BATCH_SIZE):
    stop = min(start + INFERENCE_BATCH_SIZE, N_DIAGNOSTICS)
    posterior = workflow.sample(
        conditions={"strain": diagnostic_strain[start:stop]},
        num_samples=N_POSTERIOR_SAMPLES,
    )
    posterior_batches.append(np.asarray(posterior["parameters"]))
posterior_samples = np.concatenate(posterior_batches, axis=0)
assert posterior_samples.shape == (N_DIAGNOSTICS, N_POSTERIOR_SAMPLES, 6)
assert np.isfinite(posterior_samples).all()
posterior_means = posterior_samples.mean(axis=1)
print("Posterior samples:", posterior_samples.shape)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    ax.hist(posterior_samples[0, :, i], bins=35, density=True, alpha=0.75)
    ax.axvline(true_parameters[0, i], color="black", linestyle="--", label="true")
    ax.set_title(name)
    ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("Posterior samples for one test signal")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "posterior_example.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    truth = true_parameters[:, i]
    prediction = posterior_means[:, i]
    lower = min(truth.min(), prediction.min())
    upper = max(truth.max(), prediction.max())
    ax.scatter(truth, prediction, s=18, alpha=0.6)
    ax.plot([lower, upper], [lower, upper], "k--")
    ax.set(title=name, xlabel="True value", ylabel="Posterior mean")
    ax.grid(alpha=0.2)
fig.suptitle("Parameter recovery")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "parameter_recovery.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
ranks = (posterior_samples < true_parameters[:, None, :]).sum(axis=1)
rank_u = (ranks + 0.5) / (N_POSTERIOR_SAMPLES + 1.0)
grid = np.linspace(0, 1, 300)
epsilon = np.sqrt(np.log(2.0 / 0.05) / (2.0 * N_DIAGNOSTICS))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    ax.hist(ranks[:, i], bins=10, range=(0, N_POSTERIOR_SAMPLES), edgecolor="black")
    ax.set(title=name, xlabel="Rank", ylabel="Count")
fig.suptitle("SBC rank histograms")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sbc_simple.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    sorted_ranks = np.sort(rank_u[:, i])
    ecdf = np.arange(1, N_DIAGNOSTICS + 1) / N_DIAGNOSTICS
    ax.fill_between(grid, np.maximum(0, grid - epsilon), np.minimum(1, grid + epsilon), color="lightgray")
    ax.plot(grid, grid, "k--")
    ax.step(sorted_ranks, ecdf, where="post")
    ax.set(title=name, xlabel="Normalized rank", ylabel="ECDF", xlim=(0, 1), ylim=(0, 1))
fig.suptitle("SBC empirical CDFs")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sbc_ecdf.png", dpi=180, bbox_inches="tight")
plt.show()

# ECDF difference view with pointwise 95% confidence bands
from scipy.stats import binom

difference_grid = np.linspace(0.0, 1.0, 500)
band_lower = binom.ppf(0.025, N_DIAGNOSTICS, difference_grid) / N_DIAGNOSTICS - difference_grid
band_upper = binom.ppf(0.975, N_DIAGNOSTICS, difference_grid) / N_DIAGNOSTICS - difference_grid

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for i, (ax, name) in enumerate(zip(axes.ravel(), PARAMETER_NAMES)):
    sorted_ranks = np.sort(rank_u[:, i])
    empirical_cdf = np.searchsorted(sorted_ranks, difference_grid, side="right") / N_DIAGNOSTICS
    ecdf_difference = empirical_cdf - difference_grid
    ax.fill_between(difference_grid, band_lower, band_upper, color="lightgray", label="95% confidence bands")
    ax.plot(difference_grid, ecdf_difference, color="midnightblue", label="Rank ECDF")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set(title=name, xlabel="Normalized rank statistic", ylabel="ECDF difference", xlim=(0, 1))
    ax.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle("SBC ECDF difference plots")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "sbc_ecdf_difference.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Inclination and posterior contraction

A single detector observes $h=F_+h_+ + F_\times h_\times$, with $h_+\propto(1+\cos^2\iota)/(2D)$ and $h_\times\propto\cos\iota/D$. Distance and inclination can therefore compensate for each other. Broad inclination posteriors, especially for distant signals, are an expected limitation rather than automatically a training error.

In [ ]:
prior_widths = np.array([70.0, 70.0, 1.98, 1.98, 900.0, np.pi])
normalized_widths = posterior_samples.std(axis=1) / prior_widths
true_distance = true_parameters[:, 4]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(true_distance, normalized_widths[:, 4], alpha=0.6)
axes[0].set(title="Distance uncertainty", xlabel="True distance [Mpc]", ylabel="Posterior std / prior width")
axes[1].scatter(true_distance, normalized_widths[:, 5], alpha=0.6)
axes[1].set(title="Inclination uncertainty", xlabel="True distance [Mpc]", ylabel="Posterior std / prior width")
for ax in axes:
    ax.axhline(1 / np.sqrt(12), color="black", linestyle="--", label="prior std")
    ax.grid(alpha=0.2)
axes[0].legend()
fig.suptitle("Posterior contraction")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "posterior_contraction.png", dpi=180, bbox_inches="tight")
plt.show()

true_iota = true_parameters[:, 5]
mean_iota = posterior_means[:, 5]
mean_cos_iota = np.cos(posterior_samples[:, :, 5]).mean(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(true_iota, mean_iota, c=true_distance, cmap="viridis", alpha=0.7)
axes[0].plot([0, np.pi], [0, np.pi], "k--")
axes[0].set(title="Inclination recovery", xlabel="True inclination [rad]", ylabel="Posterior mean [rad]")
scatter = axes[1].scatter(np.cos(true_iota), mean_cos_iota, c=true_distance, cmap="viridis", alpha=0.7)
axes[1].plot([-1, 1], [-1, 1], "k--")
axes[1].set(title="Recovery in cos(inclination)", xlabel="True cos(inclination)", ylabel="Posterior mean")
fig.colorbar(scatter, ax=axes, label="Distance [Mpc]")
fig.savefig(FIGURE_DIR / "inclination_recovery.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
expected_outputs = [
    MODEL_PATH,
    FIGURE_DIR / "training_loss.png",
    FIGURE_DIR / "posterior_example.png",
    FIGURE_DIR / "parameter_recovery.png",
    FIGURE_DIR / "sbc_simple.png",
    FIGURE_DIR / "sbc_ecdf.png",
    FIGURE_DIR / "sbc_ecdf_difference.png",
    FIGURE_DIR / "posterior_contraction.png",
    FIGURE_DIR / "inclination_recovery.png",
]
for path in expected_outputs:
    print(path, path.exists())
    assert path.exists()
print("Full pipeline completed successfully.")